# Fine-Tuning vs Zero-Shot Classification
### Sprint 2, Item 3

The core modeling decision for this sprint: should resumes be classified into job
categories by **fine-tuning** a model on labeled examples, or by **zero-shot
classification** using a pretrained model with no task-specific training?

This notebook runs zero-shot classification end-to-end (since it needs no labeled
data), and lays out the fine-tuning path as a documented alternative — useful if a
labeled validation set becomes available later.

First run downloads the `facebook/bart-large-mnli` model (~1.6GB) and needs internet access.


In [1]:
%run config.ipynb

Config loaded. Data will be stored under: c:\Users\RanVic\Downloads\applicant_screening_sprint2\applicant_screening_nb\data_pipeline\data


## 1. Load cleaned data

In [2]:
import pandas as pd

resumes_clean = pd.read_csv(PROCESSED_RESUMES_CSV)
job_descriptions = pd.read_csv(JOB_DESCRIPTIONS_CSV)
candidate_labels = job_descriptions[COL_CATEGORY].tolist()

print(f"Loaded {len(resumes_clean)} resumes")
print(f"Candidate labels ({len(candidate_labels)}): {candidate_labels[:5]} ...")

Loaded 2481 resumes
Candidate labels (24): ['ACCOUNTANT', 'ADVOCATE', 'AGRICULTURE', 'APPAREL', 'ARTS'] ...


## 2. Zero-shot classification
Uses `facebook/bart-large-mnli`, a model trained on natural language inference (NLI)
that can classify text against *any* set of labels supplied at inference time — no
task-specific training needed. It works by framing classification as: "does this
text entail the statement 'This example is about {label}'?" for each candidate label.

In [3]:
from transformers import pipeline
from tqdm.auto import tqdm

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
)

# Demo on a handful of resumes first (zero-shot is much slower than TF-IDF/embeddings —
# it runs one full forward pass per candidate label, per resume)
SAMPLE_SIZE = 20
sample = resumes_clean.sample(n=min(SAMPLE_SIZE, len(resumes_clean)), random_state=RANDOM_SEED)

results = []
for _, row in tqdm(sample.iterrows(), total=len(sample), desc="Classifying resumes"):
    # Truncate long resumes — BART has a 1024-token limit
    text = row["clean_text"][:2000]
    prediction = classifier(text, candidate_labels, multi_label=False)
    results.append({
        COL_ID: row[COL_ID],
        "true_category": row[COL_CATEGORY],
        "predicted_category": prediction["labels"][0],
        "confidence": prediction["scores"][0],
    })

zero_shot_results = pd.DataFrame(results)
zero_shot_results.head(10)

C:\Users\RanVic\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Classifying resumes: 100%|██████████| 20/20 [1:25:14<00:00, 255.70s/it]


,ID,true_category,predicted_category,confidence
0,46260230,INFORMATION-TECHNOLOGY,INFORMATION-TECHNOLOGY,0.401043
1,10289113,BUSINESS-DEVELOPMENT,BUSINESS-DEVELOPMENT,0.384927
2,20566550,ENGINEERING,ENGINEERING,0.540283
3,26456899,ENGINEERING,ENGINEERING,0.690640
4,17660419,FITNESS,FITNESS,0.148859
5,27409087,FINANCE,FINANCE,0.273483
6,21178545,FITNESS,HR,0.146519
7,61322296,CHEF,CHEF,0.268468
8,3547447,BANKING,BANKING,0.146247
9,23577836,ADVOCATE,ADVOCATE,0.392416


In [4]:
accuracy = (zero_shot_results["true_category"] == zero_shot_results["predicted_category"]).mean()
print(f"Zero-shot top-1 accuracy on {len(zero_shot_results)}-resume sample: {accuracy:.1%}")

Zero-shot top-1 accuracy on 20-resume sample: 85.0%


## 3. Fine-tuning: the alternative path (not run here)
Fine-tuning would mean training `bart-large-mnli` (or a smaller classifier like
`distilbert-base-uncased`) directly on labeled (resume, category) pairs from this
dataset — using the existing `Category` column as ground truth. Skeleton below shows
the shape of that approach; it isn't executed automatically since it takes
significant compute time (minutes to hours depending on hardware) and permanently
changes model weights, so it's left as an opt-in step.

In [ ]:
# --- Fine-tuning skeleton (NOT executed automatically) ---
# from sklearn.model_selection import train_test_split
# from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
#
# train_df, test_df = train_test_split(
#     resumes_clean, test_size=0.2, stratify=resumes_clean[COL_CATEGORY], random_state=RANDOM_SEED
# )
#
# tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
# model = AutoModelForSequenceClassification.from_pretrained(
#     "distilbert-base-uncased", num_labels=len(candidate_labels)
# )
#
# # ... tokenize train_df/test_df, wrap in a Dataset, then:
# training_args = TrainingArguments(
#     output_dir="./finetune_output", num_train_epochs=3, per_device_train_batch_size=8
# )
# trainer = Trainer(model=model, args=training_args, train_dataset=..., eval_dataset=...)
# trainer.train()

print("Fine-tuning skeleton above is commented out — run manually if you have time/compute to spare.")

## 4. Decision: which approach fits this project?

| | Zero-shot | Fine-tuning |
|---|---|---|
| Labeled data needed | None | Needs (resume, category) pairs — we have this via `Category`, but it's noisy (auto-derived job descriptions, not real hiring outcomes) |
| Setup time | Minutes | Hours (training + tuning) |
| Accuracy ceiling | Lower — general-purpose model | Higher — learns this dataset's specific patterns |
| Generalizes to new categories | Yes — just change the label list | No — requires retraining |

**For this project:** zero-shot is the more practical choice for a capstone timeline,
and it generalizes to new job categories without retraining — which matters since a
real screening tool will see job postings the training data never covered. Fine-tuning
is worth revisiting only if accuracy proves insufficient in the evaluation notebook
next, and a larger, cleaner labeled set becomes available.
